# 第八章：波动率微笑与曲面 / Chapter 8: Volatility Smile and Surface

## 你将学到 / What You Will Learn
- 什么是"隐含波动率" / What "implied volatility" is
- 为什么现实中波动率不是一个固定数字 / Why real-world vol is not a single number
- 什么是"波动率微笑"和"波动率偏斜" / What "vol smile" and "vol skew" mean
- 漂亮的3D波动率曲面长什么样 / What a pretty 3D vol surface looks like

---

## 8.1 先回顾：波动率是什么？ / 8.1 Quick Recap: What Is Volatility?

> **波动率** = 价格上蹿下跳的剧烈程度
>
> **Volatility** = how violently the price jumps around.

- 波动率高 → 价格变化大 → 期权更贵（因为赚大钱的可能性大）/ High vol → big moves → pricier options (bigger upside for the buyer)
- 波动率低 → 价格变化小 → 期权更便宜 / Low vol → calm moves → cheaper options

在 Black-Scholes 公式中，我们用 σ（sigma）代表波动率。

In the Black-Scholes formula, volatility is denoted σ (sigma).

## 8.2 隐含波动率 = "市场认为的波动率" / 8.2 Implied Vol = "The Market's View of Volatility"

> 问题：如果我们已经知道一个期权的市场价格，能不能**反推**出市场认为的波动率是多少？
>
> Question: given an option's market price, can we **back out** the volatility that the market is pricing in?

答案是可以！把市场价格代入 BS 公式，解方程反求 σ，得到的就叫**隐含波动率(IV)**。

Yes — plug the market price into BS and solve for σ. That σ is the **implied volatility (IV)**.

## 8.3 波动率微笑：现实打了BS公式的脸 / 8.3 The Vol Smile — Reality Slaps Black-Scholes

BS 公式假设所有行权价的波动率 σ 是一样的。

Black-Scholes assumes a single σ across every strike.

但现实中：**不同行权价对应的隐含波动率是不一样的！**

But in reality: **IV is different for different strikes!**

画出来呈 U 形（微笑）。原因：市场参与者害怕极端情况（暴涨暴跌），所以深度虚值期权的需求更大，价格偏高，反推出的 IV 也就偏高。

Plotted, it forms a U-shape — the "smile". The reason: market participants fear extreme moves (crashes and squeezes), so deep-out-of-the-money options are in high demand, they trade rich, and their implied vols are therefore lifted.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

C_GREEN, C_RED, C_ORANGE, C_BLUE, C_PURPLE = '#2ecc71', '#e74c3c', '#f39c12', '#3498db', '#9b59b6'
print('Setup done!')

## 8.4 互动实验 / 8.4 Interactive Experiment

> 拖动滑块观察：
> - **ATM波动率**: 平值点的波动率基准
> - **偏斜(Skew)**: 左边高还是右边高（负值=左偏，反映"怕暴跌"）
> - **微笑(Smile)**: U形有多弯
> - **期限效应**: 短期和长期的波动率差异
>
> Drag the sliders and watch:
> - **ATM vol** — the baseline volatility at the money
> - **Skew** — is the left side higher or the right? (Negative = left-heavy = fear of crashes)
> - **Smile** — how curved the U-shape is
> - **Term effect** — how short-dated vol differs from long-dated vol

In [ ]:
def vol_surface(atm_vol=20, skew=-2.0, smile=1.5, term_effect=5.0):
    plt.close('all')
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

    K_arr = np.linspace(70, 130, 100)
    moneyness = (K_arr - 100) / 100  # ATM=100

    iv = atm_vol + skew * moneyness * 100 + smile * (moneyness * 100)**2 / 100

    # Left: smile
    ax = axes[0]
    ax.plot(K_arr, iv, color=C_BLUE, lw=3)
    ax.fill_between(K_arr, iv, atm_vol, where=(iv > atm_vol), color=C_BLUE, alpha=0.1)
    ax.axhline(y=atm_vol, color=C_ORANGE, ls='--', lw=1.5, label=f'ATM vol = {atm_vol}%')
    ax.axvline(x=100, color=C_PURPLE, ls=':', alpha=0.5, label='ATM (K=S)')
    ax.set_xlabel('Strike K (spot=100)', fontsize=11)
    ax.set_ylabel('Implied Vol IV (%)', fontsize=11)
    ax.set_title('Volatility Smile', fontsize=14, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)
    ax.annotate('OTM Put\n(crash fear)',
                xy=(75, iv[np.argmin(np.abs(K_arr - 75))]),
                xytext=(60, atm_vol + 10), fontsize=9, ha='center', color=C_RED,
                arrowprops=dict(arrowstyle='->', color=C_RED))
    ax.annotate('OTM Call',
                xy=(125, iv[np.argmin(np.abs(K_arr - 125))]),
                xytext=(135, atm_vol + 10), fontsize=9, ha='center', color=C_GREEN,
                arrowprops=dict(arrowstyle='->', color=C_GREEN))

    # Right: 3D surface
    ax3d = fig.add_subplot(122, projection='3d')
    axes[1].remove()
    T_arr = np.linspace(0.1, 2.0, 25)
    Kg, Tg = np.meshgrid(K_arr, T_arr)
    mng = (Kg - 100) / 100
    iv_surf = atm_vol + skew * mng * 100 + smile * (mng * 100)**2 / 100 - term_effect * (Tg - 1.0)

    ax3d.plot_surface(Kg, Tg, iv_surf, cmap=cm.coolwarm, alpha=0.85, edgecolor='none')
    ax3d.set_xlabel('Strike K', fontsize=9)
    ax3d.set_ylabel('T (years)', fontsize=9)
    ax3d.set_zlabel('IV (%)', fontsize=9)
    ax3d.set_title('3D Vol Surface', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.show()

    print(f'  ATM vol={atm_vol}%, skew={skew}, smile={smile}')
    if skew < -1:
        print('  Negative skew -> market fears crashes (low-K IV high)')
    elif skew > 1:
        print('  Positive skew -> market fears rallies')
    else:
        print('  Near-zero skew -> roughly symmetric')

interact(vol_surface,
         atm_vol=FloatSlider(value=20, min=10, max=50, step=1, description='ATM vol(%):'),
         skew=FloatSlider(value=-2.0, min=-10, max=5, step=0.5, description='Skew:'),
         smile=FloatSlider(value=1.5, min=0, max=10, step=0.5, description='Smile:'),
         term_effect=FloatSlider(value=5.0, min=-5, max=15, step=0.5, description='Term effect:'));

## 8.5 小结 / 8.5 Chapter Recap

| 概念 / Concept | 一句话 / One-liner |
|------|--------|
| 隐含波动率 IV / Implied vol (IV) | 从市场价格反推出的"市场认为的波动率" / The vol the market is pricing in, backed out from option prices |
| 波动率微笑 / Vol smile | 不同行权价的IV不同，画出来像微笑 / IV differs by strike; plotted, it looks like a smile |
| 波动率偏斜 / Vol skew | 左边高=怕暴跌（股票市场常见）/ Left side higher = fear of a crash (typical in equity markets) |
| 波动率曲面 / Vol surface | 行权价+到期时间 → 3D的IV分布 / IV as a 3D function of strike and maturity |

> **下一章：期权策略组合** —— 怎么把几个期权拼起来做更聪明的交易？
>
> **Next chapter: option strategy combos** — stitching multiple options together for smarter trades.